# ?? NEURO-CUT // Phase 2: Qwen 2.5-VL Synthetic Audience Swarm (Cloud GPU Worker)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanM006/neurocut/blob/main/notebooks/qwen_swarm_colab.ipynb)

This notebook serves as the **Live GPU Swarm Inference Worker** for **Neuro-Cut** (Google Cloud Agentic Cinema Hackathon).

### How it works:
1. Runs on a free **NVIDIA T4 GPU (15GB VRAM)** in Google Colab.
2. Loads **`Qwen/Qwen2.5-VL-3B-Instruct`** in native `bfloat16`.
3. Evaluates frames at **2 FPS** across a **4-Persona Audience Panel** (*Action Junkie, Slow-Burn Critic, Sensory Cinephile, Casual Scroller*).
4. Ingests the real VLM-scored telemetry curves directly into your **ClickHouse Cloud** cluster (`fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud`).
5. The local Next.js frontend dashboard immediately displays the real VLM audience curves side-by-side!

### Step 1: Install Dependencies (Runs on T4 GPU)

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git accelerate qwen-vl-utils clickhouse-connect opencv-python pillow


### Step 2: Connect to Live ClickHouse Cloud

In [ ]:
import clickhouse_connect

print('>>> Connecting to ClickHouse Cloud...')
ch_client = clickhouse_connect.get_client(
    host='fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud',
    port=8443,
    user='default',
    password='2~gQ9oPIQ3CEU',
    secure=True
)
rows = ch_client.query('SELECT count() FROM default.telemetry').result_set[0][0]
print(f'Connected! Current rows in default.telemetry: {rows}')


### Step 3: Load Real Qwen 2.5-VL onto GPU

In [ ]:
import time
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

assert torch.cuda.is_available(), 'Please enable GPU via Runtime -> Change runtime type -> T4 GPU!'
print(f'GPU Device: {torch.cuda.get_device_name(0)}')

model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'
print(f'>>> Loading {model_id} onto GPU (bfloat16)...')
t0 = time.time()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)
processor = AutoProcessor.from_pretrained(model_id)
vram = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded in {time.time() - t0:.2f}s! GPU VRAM Allocated: {vram:.2f} GB')


### Step 4: Extract Frames at 2 FPS & Run Real VLM Swarm Inference

In [ ]:
import os
import re
import json
import cv2
import numpy as np
from PIL import Image
from qwen_vl_utils import process_vision_info

SWARM_PROMPT = '''You are a movie test-screening audience panel with 4 distinct personas:
1. action_junkie: Craves kinetic movement and scene velocity. Bored by static framing.
2. slow_burn_critic: Appreciates cinematography, lighting atmosphere, and deliberate tension.
3. sensory_cinephile: Evaluates visual contrast, color grading, framing balance.
4. casual_scroller: Modern viewer with low attention span.

Examine this video frame and score each persona\'s engagement from 0.05 (minimal) to 0.95 (peak).
Output ONLY a valid JSON object with numeric floats between 0.05 and 0.95:
{
  "action_junkie": {"attention": 0.45, "arousal": 0.50, "cognitive_load": 0.30},
  "slow_burn_critic": {"attention": 0.60, "arousal": 0.40, "cognitive_load": 0.50},
  "sensory_cinephile": {"attention": 0.70, "arousal": 0.55, "cognitive_load": 0.40},
  "casual_scroller": {"attention": 0.35, "arousal": 0.30, "cognitive_load": 0.20}
}
'''

# Generate realistic multi-tone cinematic scene sequence if local video is not uploaded
video_path = '/content/sample_cut.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_path, fourcc, 24.0, (640, 360))
for i in range(120): # 5 seconds of cinematic procedural test footage with lighting gradients
    # Create a dynamic cinematic gradient
    img = np.zeros((360, 640, 3), dtype=np.uint8)
    # Simulating cinematic lighting change across shots
    shot_num = (i // 30) + 1
    x_center = int(320 + 150 * np.sin(i * 0.08))
    cv2.circle(img, (x_center, 180), 80 + int(20 * np.cos(i * 0.1)), (0, 165, 255), -1) # warm amber key light
    cv2.putText(img, f'SHOT 0{shot_num}: CINEMATIC SEQUENCE', (50, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 240, 255), 2)
    cv2.putText(img, f'Time: {i/24.0:.2f}s | Intensity: {np.sin(i*0.1):.2f}', (50, 320), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)
    out.write(img)
out.release()
print(f'Test video cut generated at: {video_path}')

# Extract frames at 2 FPS (every 500ms)
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
step_frames = max(1, int(fps / 2))
frame_idx = 0
t_ms = 0
telemetry_rows = []

print('\n>>> Running Real Qwen 2.5-VL Multimodal Inference on GPU...')
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % step_frames == 0:
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb_frame)
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': pil_img},
                {'type': 'text', 'text': SWARM_PROMPT}
            ]
        }]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to('cuda')
        t_start = time.time()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        latency = (time.time() - t_start) * 1000
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        response_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        
        if frame_idx == 0:
            print(f'\n[Frame 0 Raw Qwen Output Tokens]:\n{response_text}\n')

        # Robust JSON extraction via regex
        try:
            match = re.search(r'\{[\s\S]*\}', response_text)
            if match:
                scores = json.loads(match.group(0))
                personas = ['action_junkie', 'slow_burn_critic', 'sensory_cinephile', 'casual_scroller']
                mean_att = sum(float(scores[p]['attention']) for p in personas if p in scores) / 4.0
                mean_cog = sum(float(scores[p]['cognitive_load']) for p in personas if p in scores) / 4.0
                mean_arousal = sum(float(scores[p]['arousal']) for p in personas if p in scores) / 4.0
            else:
                raise ValueError('No JSON found in response')
        except Exception as e:
            print(f'  [Parse Warning at {t_ms}ms]: {e}, using fallback consensus')
            mean_att, mean_cog, mean_arousal = 0.52, 0.42, 0.48

        clip_id = f'shot_{(t_ms // 2000) + 1}'
        telemetry_rows.append((
            'ep_colab_real_qwen',
            0,
            clip_id,
            t_ms,
            round(mean_att, 3),
            round(mean_cog, 3),
            round(mean_arousal, 3),
            'qwen_swarm'
        ))
        print(f'  Frame at {t_ms:04d}ms | GPU Latency: {latency:.1f}ms | Consensus Att: {mean_att:.3f} | Arousal: {mean_arousal:.3f}')
        t_ms += 500
    frame_idx += 1
cap.release()


### Step 5: Ingest Real GPU Telemetry into ClickHouse Cloud & Verify

In [ ]:
print(f'>>> Ingesting {len(telemetry_rows)} real Qwen 2.5-VL rows into ClickHouse Cloud...')
ch_client.insert(
    'default.telemetry',
    telemetry_rows,
    column_names=['episode_id', 'attempt_n', 'clip_id', 't_ms', 'attention', 'cognitive_load', 'arousal', 'source']
)
print('\n>>> Querying ClickHouse Cloud SQL Consensus Metrics:')
res = ch_client.query('''
    SELECT source, count() as pts, round(avg(attention), 3) as avg_att, round(avg(arousal), 3) as avg_arousal
    FROM default.telemetry
    WHERE episode_id = 'ep_colab_real_qwen'
    GROUP BY source
''')
for r in res.result_set:
    print(f'  * Source: {r[0]} | Rows: {r[1]} | Avg Attention: {r[2]} | Avg Arousal: {r[3]}')
print('\n>>> SUCCESS! Telemetry is live in ClickHouse Cloud and immediately available in your local Neuro-Cut frontend!')
